In [1]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

In [2]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
# !pip install pandas
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


In [14]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
# 只对数值列填充均值
inputs = inputs.fillna(inputs.select_dtypes(include=['number']).mean())
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


In [15]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


In [16]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

## 创建包含更多行和列的原始数据集。
### 1. 删除缺失值最多的列。
### 2. 将预处理后的数据集转换为张量格式。


In [1]:
import pandas as pd
import numpy as np
import torch

# ===================== 1. 创建包含更多行和列的模拟数据集 =====================
np.random.seed(42)
data = pd.DataFrame({
    # 数值型特征
    'NumRooms': np.random.choice([1, 2, 3, 4, 5, np.nan], size=50, p=[0.1, 0.2, 0.3, 0.2, 0.15, 0.05]),
    'SquareFeet': np.random.choice([50, 60, 70, 80, 90, 100, np.nan], size=50,
                                   p=[0.1, 0.15, 0.2, 0.2, 0.15, 0.15, 0.05]),
    'Floor': np.random.choice([1, 2, 3, 4, 5, np.nan], size=50, p=[0.2, 0.2, 0.2, 0.2, 0.15, 0.05]),
    'YearBuilt': np.random.choice([2000, 2005, 2010, 2015, 2020, np.nan], size=50, p=[0.1, 0.2, 0.2, 0.2, 0.2, 0.1]),
    # 类别型特征
    'Alley': np.random.choice(['Pave', 'Gravel', np.nan], size=50, p=[0.4, 0.3, 0.3]),
    'Heating': np.random.choice(['Gas', 'Electric', 'Oil', np.nan], size=50, p=[0.4, 0.3, 0.15, 0.15]),
    'Condition': np.random.choice(['Good', 'Fair', 'Poor', np.nan], size=50, p=[0.4, 0.3, 0.1, 0.2]),
    # 目标变量
    'Price': np.random.uniform(50, 200, size=50)
})

print("原始数据集（前5行）：")
print(data.head())
print("\n各列缺失值数量：")
missing_counts = data.isnull().sum()
print(missing_counts)

# ===================== 2. 删除缺失值最多的列 =====================
max_missing_col = missing_counts.idxmax()
print(f"\n缺失值最多的列：{max_missing_col}（缺失数量：{missing_counts[max_missing_col]}）")
data_processed = data.drop(columns=[max_missing_col])
print(f"\n删除 {max_missing_col} 后的数据集形状：{data_processed.shape}")

# ===================== 3. 预处理（核心修复：确保全为数值类型） =====================
# 步骤1：分离数值列和类别列
numeric_cols = data_processed.select_dtypes(include=['number']).columns
categorical_cols = data_processed.select_dtypes(include=['object']).columns

# 步骤2：填充缺失值（数值列用均值，类别列用众数）
# 数值列填充+强制转换为float
data_processed[numeric_cols] = data_processed[numeric_cols].fillna(data_processed[numeric_cols].mean()).astype(float)
# 类别列填充（确保无NaN）
for col in categorical_cols:
    # 兜底：如果众数为空，用固定字符串填充
    fill_value = data_processed[col].mode()[0] if not data_processed[col].mode().empty else 'Unknown'
    data_processed[col] = data_processed[col].fillna(fill_value).astype(str)  # 强制转为字符串，避免混合类型

# 步骤3：类别列独热编码（彻底转为数值）
data_processed = pd.get_dummies(data_processed, columns=categorical_cols, drop_first=True, dtype=int)

# 步骤4：强制将所有列转为float32（关键！避免object类型残留）
data_processed = data_processed.astype(np.float32)

# 步骤5：转换为PyTorch张量
tensor_data = torch.tensor(data_processed.values, dtype=torch.float32)

# ===================== 验证结果 =====================
print("\n预处理后数据类型（确保无object）：")
print(data_processed.dtypes)
print("\n预处理后的数据集（前5行）：")
print(data_processed.head())
print(f"\n转换后的张量形状：{tensor_data.shape}")
print(f"张量前3行：\n{tensor_data[:3]}")

原始数据集（前5行）：
   NumRooms  SquareFeet  Floor  YearBuilt   Alley Heating Condition  \
0       3.0         NaN    1.0        NaN  Gravel     Gas      Good   
1       NaN        90.0    4.0     2005.0    Pave     Gas      Fair   
2       4.0       100.0    2.0     2005.0    Pave     nan      Fair   
3       3.0       100.0    3.0     2010.0     nan     Gas      Fair   
4       2.0        80.0    5.0        NaN  Gravel     Gas      Poor   

        Price  
0  125.470439  
1  178.473476  
2  148.804045  
3   74.440164  
4   60.585312  

各列缺失值数量：
NumRooms      3
SquareFeet    2
Floor         2
YearBuilt     5
Alley         0
Heating       0
Condition     0
Price         0
dtype: int64

缺失值最多的列：YearBuilt（缺失数量：5）

删除 YearBuilt 后的数据集形状：(50, 7)

预处理后数据类型（确保无object）：
NumRooms          float32
SquareFeet        float32
Floor             float32
Price             float32
Alley_Pave        float32
Alley_nan         float32
Heating_Gas       float32
Heating_Oil       float32
Heating_nan       float32
C